# Observability & Governance

You cannot operate a system you cannot see. **Azure Monitor** is the umbrella product that gathers everything emitted by Azure resources and applications — metrics, logs, traces — and gives you the dashboards, alerts, and workbooks to act on it. Around it sit **Application Insights** for application performance, **Azure Advisor** for recommendations, and **Service Health** for Microsoft's view of the platform.

The shape: every Azure resource emits a few **metrics** for free, a structured **Activity Log** of management events for free, and optional **resource logs** you route via diagnostic settings. Applications add **traces and custom telemetry** via Application Insights. All of it lands in **Log Analytics workspaces**, queryable with **KQL** — the same query language across every Azure data plane that emits logs.

## Azure Monitor — the umbrella

**Azure Monitor** is not one service — it is the brand for a collection of pipes that collect, store, and act on telemetry. The three data types:

- **Metrics** — time-series of numeric values, low-latency (sub-minute), retained for 93 days. CPU percentage, request count, queue depth. Every Azure resource emits a built-in set; you can also publish custom metrics from code.
- **Logs** — structured records (rows with columns) of events. Resource logs from Azure services, application telemetry, Activity Log, custom logs. Stored in **Log Analytics workspaces**.
- **Traces** — distributed traces, OpenTelemetry-compatible, captured by Application Insights. Stored in Log Analytics as a special schema.

The umbrella exposes:

- **Metrics Explorer** — charting and ad-hoc analysis over metrics.
- **Log Analytics** — KQL queries over logs and traces.
- **Alerts** — threshold-based or query-based rules that fire to action groups.
- **Workbooks** — interactive, parameterised reports combining metrics, logs, and visualisations. The right primitive for SRE runbooks and exec dashboards.
- **Application Insights** — APM built on the same plumbing.

AWS comparison: Azure Monitor ≈ CloudWatch (metrics + logs + alarms + dashboards) + X-Ray (tracing). Azure consolidates the experience; AWS has separate consoles.

## Log Analytics workspaces & KQL

**Log Analytics workspaces** are the storage and query engine for logs and traces. A workspace is a regional resource, billed by GB ingested and GB retained.

Inside the workspace, data lives in **tables** with a schema: `AzureActivity`, `AzureDiagnostics`, `AppRequests`, `AppExceptions`, `Heartbeat`, custom tables, and so on. You query with **Kusto Query Language (KQL)** — the same language Sentinel, Defender, and Application Insights all use.

A few KQL essentials worth keeping in your head:

```kql
AppRequests
| where TimeGenerated > ago(1h)
| where Success == false
| summarize count() by ResultCode, bin(TimeGenerated, 5m)
| render timechart
```

Pipeline syntax (`|`) chains operators; tabular results flow left to right. The grammar is small: `where` filters, `project` selects columns, `extend` adds calculated columns, `summarize` aggregates, `join` joins, `render` visualises.

**Workspace design.** A common pattern is one workspace per environment (or per region per environment) shared across many resource groups. Centralise *security* logs into a Sentinel workspace; keep *operational* logs in workload-aligned workspaces — sometimes the same, sometimes split for cost and access reasons. **Workspace-based Application Insights** (the default since 2020) stores telemetry in a Log Analytics workspace too, so APM and infra logs sit in the same store and KQL can join across them.

**Data export & basic logs** — for high-volume logs you don't need to query interactively (verbose audit, network flows), the **Basic Logs** tier is far cheaper but limits the operators you can run. For long-term retention beyond a year, **archive tier** drops cost further and accepts a rehydrate step before query.

## Diagnostic Settings, Activity Log, Resource Logs

Every Azure resource emits two kinds of logs by default — and one of them you have to ask for.

- **Activity Log** — subscription-level management plane log. Every ARM call (who created what, when, from where). Free to read, retained 90 days. Sent to a Log Analytics workspace via a diagnostic setting if you want longer retention or KQL joins.
- **Resource logs** — data plane / per-resource logs (`StorageRead`, `AppServiceHTTPLogs`, `KeyVaultAuditEvent`, `AzureSQLAuditEvent`, etc.). **Off by default**. Turn them on via **Diagnostic Settings** on each resource.

A diagnostic setting on a resource picks log categories and routes them to one or more destinations:

- **Log Analytics workspace** — the default; queryable with KQL.
- **Storage account** — cheap archival.
- **Event Hub** — stream to a third-party SIEM (Splunk, Elastic) or to your own consumer.
- **Partner integration** — Datadog, Logz.io, etc.

**Diagnostic Setting** policies (Azure Policy with DeployIfNotExists effect) enforce that every new resource of a given type gets a setting created automatically. This is the operational pattern — *don't* expect engineers to remember to turn diagnostic settings on. Wire them through policy.

The terminology trips people up: **Activity Log** is *management plane*; **Resource Logs** are *data plane*. If a question is about who deleted a resource, the answer is in Activity Log. If a question is about who read a secret, the answer is in Key Vault's resource log — only if you turned it on.

## Application Insights

**Application Insights (App Insights)** is Azure's APM (application performance monitoring). It captures requests, dependencies (DB calls, HTTP calls), exceptions, traces, custom events, and metrics from your application — with **auto-instrumentation** for .NET, Java, Node, Python, and the App Service / Functions / Container Apps runtimes.

Three primitives in the schema:

- **Requests / Dependencies** — every incoming and outgoing call, with duration, success/failure, and operation ID. Used to build distributed traces.
- **Exceptions / Traces** — exceptions caught (and ones surfaced by the runtime) plus arbitrary log messages from your code.
- **Custom events / Custom metrics** — application-defined business telemetry.

**Distributed tracing** stitches operations together by correlation IDs (`operation_Id`, `operation_ParentId`). When request A in service X calls service Y, App Insights propagates the IDs via W3C Trace Context headers; the resulting trace shows the full call tree across services in the Transaction Search view.

**Sampling** is on by default — Application Insights drops some percentage of low-value traces to keep ingestion costs sane. **Adaptive sampling** auto-tunes the rate; **fixed-rate sampling** lets you pin it. For high-traffic services, sampling is necessary — without it, you ingest petabytes of trivial successes. Failures, slow requests, and exceptions are sampled less aggressively.

**Availability tests** are synthetic probes that hit your endpoint from multiple Azure regions and alert on failure. **Standard tests** are the modern replacement for the legacy URL ping tests; they support custom headers, auth, and content validation.

**Live Metrics** is the real-time stream — open it during a deploy or incident and you see traffic, failures, and dependencies in true real time, without waiting for ingestion lag.

AWS comparison: App Insights ≈ CloudWatch RUM + X-Ray + CloudWatch Synthetics combined. The auto-instrumentation story on Azure is more polished than the X-Ray equivalent.

## Alerts and action groups

Alerts in Azure Monitor come in four flavours, and you will use all four:

- **Metric alerts** — threshold on a metric (CPU > 80% for 5 minutes). Cheapest and lowest latency.
- **Log alerts** — a KQL query runs on schedule; alert fires when the result crosses a threshold.
- **Activity Log alerts** — fire on specific management events (`Microsoft.KeyVault/vaults/delete`).
- **Smart Detection / anomaly alerts** — Application Insights detects unusual behaviour (response time spike, dependency failure spike) without you defining the threshold.

Alerts fire to **action groups** — reusable collections of notification destinations and response actions:

- Email, SMS, push to the Azure mobile app, voice call.
- Webhook to PagerDuty, Opsgenie, or a custom service.
- Logic App, Function, Automation runbook, ITSM ticket (ServiceNow, etc.).
- **Event Grid** topic for downstream processing.

Pattern: have a small number of action groups (`ag-prod-critical`, `ag-prod-warning`, `ag-noncrit-team`) and bind every alert to the right one. Don't create per-alert action groups — you'll end up unable to change pager routing without touching dozens of resources.

**Alert processing rules** sit on top — suppress alerts during maintenance windows, route different severities differently, override action groups in bulk.

## Workbooks and dashboards

Two ways to visualise Azure Monitor data, and they serve different purposes:

- **Azure Dashboards** — pinned tiles (metric chart, log query, markdown). Per-user or shared. Best for a *small* always-on summary view.
- **Workbooks** — interactive, parameterised reports — a runbook in document form, with KQL queries, charts, tabbed sections, drill-down links. Microsoft ships hundreds of templates for specific scenarios (AKS cluster health, Storage account performance, Cost analysis). Workbooks are the right primitive for SRE runbooks, exec rollups, and incident-investigation guides.

Workbooks beat dashboards for almost every non-trivial use case — they support parameters, conditional sections, and complex layouts that dashboards don't.

## Azure Advisor and Service Health

Two free services that earn a sentence:

**Azure Advisor** analyses your subscription configuration and emits recommendations across five categories: Reliability, Security, Performance, Cost, Operational Excellence. "Right-size or shut down underutilised VMs." "Enable soft delete on this Key Vault." "Switch this SQL DB to a Hyperscale tier for X% lower cost." Review it weekly; the cost recommendations alone often pay for the time.

**Azure Service Health** is Microsoft's status view personalised to your subscriptions and regions. Three tabs:

- **Service issues** — ongoing incidents that affect your resources.
- **Planned maintenance** — upcoming patches and reboots; configure alerts.
- **Health advisories** — non-incident notices (deprecations, certificate updates).

Wire Service Health alerts into the same action groups as your metric alerts so the on-call team learns about Azure-side issues without checking the portal.

AWS comparison: Advisor ≈ Trusted Advisor; Service Health ≈ AWS Health Dashboard.

In [ ]:
# Wire up a workspace, diagnostic setting, and a couple of alerts.

RG=rg-obs-demo
WS=law-foundations
az group create -n $RG -l eastus

# 1. Log Analytics workspace.
az monitor log-analytics workspace create -g $RG -n $WS -l eastus
WSID=$(az monitor log-analytics workspace show -g $RG -n $WS --query id -o tsv)

# 2. Send a Key Vault's resource logs to the workspace.
az monitor diagnostic-settings create \
  --name to-law \
  --resource $(az keyvault show -n <kv-name> --query id -o tsv) \
  --workspace $WSID \
  --logs '[{"categoryGroup":"audit","enabled":true}]'

# 3. Action group with email + webhook.
az monitor action-group create -g $RG -n ag-prod-critical \
  --short-name prod-crit \
  --action email oncall oncall@contoso.com \
  --action webhook pagerduty https://events.pagerduty.com/v2/enqueue

# 4. Metric alert: VM CPU > 90% for 10 minutes.
az monitor metrics alert create -g $RG -n alert-vm-cpu-high \
  --scopes /subscriptions/<sub>/resourceGroups/$RG/providers/Microsoft.Compute/virtualMachines/<vm> \
  --condition "avg Percentage CPU > 90" \
  --window-size 10m --evaluation-frequency 1m \
  --action ag-prod-critical

# 5. Log alert: any failed Key Vault read in the last 5 minutes.
az monitor scheduled-query create -g $RG -n alert-kv-failed-read \
  --scopes $WSID \
  --condition "count 'KeyVaultAuditEvent | where ResultType != \"Success\" | where OperationName == \"SecretGet\"' > 0" \
  --window-size 5m --evaluation-frequency 5m \
  --action ag-prod-critical

## Putting it together

A production observability + governance setup:

1. **One Log Analytics workspace per environment** (or per region+environment), provisioned through your landing-zone IaC.
2. **Diagnostic Setting policy** with DeployIfNotExists effect on every resource type that has logs, pointing at the right workspace.
3. **Application Insights workspace-based**, one per service or per environment, with auto-instrumentation enabled on the runtime (App Service / Functions / AKS), adaptive sampling on.
4. **A small action-group catalogue** — `critical` (page on-call), `warning` (Teams channel), `info` (email digest) — referenced by every alert.
5. **Alerts as code** — metric/log/activity alerts defined in Bicep or Terraform alongside the resource they monitor. Smart Detection on every App Insights instance.
6. **Workbooks**, not dashboards, for the SRE runbooks; pinned dashboards only for always-on summaries.
7. **Advisor + Service Health alerts** wired to the same action groups so platform issues and your own incidents share a channel.

Observability done right is what lets the rest of this series stay simple — when something breaks (and something always breaks), you find out fast, you see the blast radius, and you have the data to fix the root cause instead of guessing. DevOps, HA/DR, and migration come next.